# Human vs Machine Translation Detection Codebase

This Jupyter Notebook is designed to evaluate and compare machine translation outputs against human translations using various metrics. It processes a CSV file containing source texts, human translations, and translations from multiple machine translation models (e.g., Claude, Gemini, Gemma, etc.).

## Workflow Overview

1. **Installation and Imports**: Installs necessary libraries (pandas, sacrebleu, evaluate, unbabel-comet, torch) and imports required modules for data handling, evaluation metrics, and progress tracking.

2. **Configuration**: Defines file paths, column names, model lists, and parameters for evaluation metrics (BLEU, BLEURT, COMET), including batch sizes for efficient processing.

3. **Helper Functions**: Includes utility functions for text cleaning, validation, sentence-level BLEU scoring, and batching lists.

4. **Data Loading**: Reads the input CSV, checks for required columns, and cleans text data to ensure validity.

5. **BLEU Scoring**: Computes sentence-level BLEU scores for each machine translation model against the human translations.

6. **BLEURT Scoring**: Loads the BLEURT model and computes quality scores in batches for valid translations.

7. **COMET Scoring**: Downloads and loads the COMET model, then predicts quality scores using source, machine translation, and reference texts.

8. **Output**: Saves the updated DataFrame with computed scores to a new CSV file.

This notebook facilitates quantitative analysis to detect differences between human and machine-generated translations, aiding in translation quality assessment and model comparison. Ensure the input CSV path is set correctly before running.

In [ ]:
!pip install pandas sacrebleu evaluate unbabel-comet torch

In [ ]:
import os
import math
import pandas as pd
from tqdm import tqdm

import sacrebleu
import evaluate

from comet import download_model, load_from_checkpoint



In [ ]:
INPUT_CSV = "" # Path to the input CSV file containing the translations and source texts.
OUTPUT_CSV = "eu_dgt_with_scores.csv"

HUMAN_COL = "human translation"
SOURCE_COL = "source text"
SOURCE_LANG_COL = "source language"

MODEL_COLS = [
    "claude",
    "gemini",
    "gemma",
    "llamma",
    "nllb",
    "openai",
    "qwen",
]

# COMET model
COMET_MODEL_NAME = "Unbabel/wmt22-comet-da"

# BLEURT checkpoint
BLEURT_CHECKPOINT = "BLEURT-20"

# Batch sizes
BLEURT_BATCH_SIZE = 32
COMET_BATCH_SIZE = 64



In [ ]:
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def is_valid_text(x):
    return isinstance(x, str) and len(x.strip()) > 0


def sentence_bleu_score(pred, ref):
    """
    Sentence-level BLEU using sacrebleu with smoothing.
    Returns float.
    """
    if not is_valid_text(pred) or not is_valid_text(ref):
        return None
    try:
        # sentence_score returns score out of 100
        score = sacrebleu.sentence_bleu(
            pred,
            [ref],
            smooth_method="exp"
        ).score
        return float(score)
    except Exception:
        return None


def chunk_list(lst, batch_size):
    for i in range(0, len(lst), batch_size):
        yield lst[i:i + batch_size]



In [ ]:
df = pd.read_csv(INPUT_CSV)

required_columns = [HUMAN_COL, SOURCE_COL, SOURCE_LANG_COL] + MODEL_COLS
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Clean text columns
for col in required_columns:
    df[col] = df[col].apply(safe_str)

print(f"Loaded {len(df)} rows from {INPUT_CSV}")



In [ ]:
print("\nComputing BLEU...")
for model_col in MODEL_COLS:
    bleu_col = f"{model_col}_bleu"
    scores = []

    for pred, ref in tqdm(
        zip(df[model_col], df[HUMAN_COL]),
        total=len(df),
        desc=f"BLEU: {model_col}"
    ):
        scores.append(sentence_bleu_score(pred, ref))

    df[bleu_col] = scores

In [ ]:
print("\nLoading BLEURT...")
bleurt = evaluate.load("bleurt", checkpoint=BLEURT_CHECKPOINT)

print("Computing BLEURT...")
for model_col in MODEL_COLS:
    bleurt_col = f"{model_col}_bleurt"
    all_scores = [None] * len(df)

    valid_indices = []
    predictions = []
    references = []

    for idx, (pred, ref) in enumerate(zip(df[model_col], df[HUMAN_COL])):
        if is_valid_text(pred) and is_valid_text(ref):
            valid_indices.append(idx)
            predictions.append(pred)
            references.append(ref)

    for batch_indices in tqdm(
        list(chunk_list(list(range(len(valid_indices))), BLEURT_BATCH_SIZE)),
        desc=f"BLEURT: {model_col}"
    ):
        batch_preds = [predictions[i] for i in batch_indices]
        batch_refs = [references[i] for i in batch_indices]

        try:
            batch_scores = bleurt.compute(
                predictions=batch_preds,
                references=batch_refs
            )["scores"]
        except Exception as e:
            print(f"BLEURT batch failed for {model_col}: {e}")
            batch_scores = [None] * len(batch_preds)

        for local_i, score in enumerate(batch_scores):
            original_idx = valid_indices[batch_indices[local_i]]
            all_scores[original_idx] = float(score) if score is not None else None

    df[bleurt_col] = all_scores



In [ ]:
print("\nLoading COMET...")
comet_model_path = download_model(COMET_MODEL_NAME)
comet_model = load_from_checkpoint(comet_model_path)

print("Computing COMET...")
for model_col in MODEL_COLS:
    comet_col = f"{model_col}_comet"
    all_scores = [None] * len(df)

    valid_indices = []
    comet_data = []

    for idx, row in df.iterrows():
        mt = row[model_col]
        ref = row[HUMAN_COL]
        src = row[SOURCE_COL]

        if is_valid_text(mt) and is_valid_text(ref) and is_valid_text(src):
            valid_indices.append(idx)
            comet_data.append({
                "src": src,
                "mt": mt,
                "ref": ref,
            })

    if len(comet_data) == 0:
        df[comet_col] = all_scores
        continue

    try:
        comet_output = comet_model.predict(
            comet_data,
            batch_size=COMET_BATCH_SIZE,
            gpus=1 if hasattr(comet_model, "predict") else 0
        )
        comet_scores = comet_output.scores
    except Exception as e:
        print(f"COMET failed for {model_col}: {e}")
        comet_scores = [None] * len(comet_data)

    for idx_in_list, score in enumerate(comet_scores):
        original_idx = valid_indices[idx_in_list]
        all_scores[original_idx] = float(score) if score is not None else None

    df[comet_col] = all_scores




In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nDone. Saved scored file to: {OUTPUT_CSV}")